In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, random, shutil, os
from tqdm import tqdm

# ====== EDIT THESE PATHS TO MATCH YOUR DRIVE ======
SAMPLE_IMAGES_DIR = Path("/content/drive/MyDrive/BarBeR_sample/images")
VIA_JSON_DIR      = Path("/content/drive/MyDrive/BarBeR_sample/Annotations/VIA")

# Output dataset (YOLO format)
OUT_DIR = Path("/content/drive/MyDrive/BarBeR_yolo_sample")
IMG_OUT = OUT_DIR / "images"
LBL_OUT = OUT_DIR / "labels"
IMG_OUT.mkdir(parents=True, exist_ok=True)
LBL_OUT.mkdir(parents=True, exist_ok=True)

print("Sample images:", len(list(SAMPLE_IMAGES_DIR.glob("*.jpg"))))
print("VIA json files:", len(list(VIA_JSON_DIR.glob("*.json"))))
print("Output:", OUT_DIR)


Mounted at /content/drive
Sample images: 789
VIA json files: 12
Output: /content/drive/MyDrive/BarBeR_yolo_sample


In [4]:
sample_imgs = sorted(list(SAMPLE_IMAGES_DIR.glob("*.jpg")))
assert len(sample_imgs) > 0, "No JPG images found in SAMPLE_IMAGES_DIR"

# Copy images into output/images
for p in tqdm(sample_imgs, desc="Copying images"):
    shutil.copy2(p, IMG_OUT / p.name)

print("Copied:", len(list(IMG_OUT.glob('*.jpg'))))


Copying images: 100%|██████████| 789/789 [10:41<00:00,  1.23it/s]

Copied: 789


In [5]:
#Parsing VIA JSON + filter to sampled filenames
import PIL.Image as Image

# Helper: polygon -> bbox
def poly_to_bbox(xs, ys):
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    return x_min, y_min, x_max, y_max

# Helper: bbox -> yolo (normalized)
def bbox_to_yolo(xmin, ymin, xmax, ymax, w, h):
    bw = xmax - xmin
    bh = ymax - ymin
    xc = xmin + bw/2
    yc = ymin + bh/2
    return xc/w, yc/h, bw/w, bh/h

# We will detect "barcode" only (single class)
CLASS_ID = 0

# Map sample filenames for quick lookup
sample_names = set([p.name for p in IMG_OUT.glob("*.jpg")])

# Build a dict: image_name -> list of yolo lines
labels_map = {name: [] for name in sample_names}

json_files = sorted(VIA_JSON_DIR.glob("*.json"))
kept_regions = 0

for jf in tqdm(json_files, desc="Reading VIA JSONs"):
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    # VIA formats vary; handle common structures
    # Most VIA exports have dict under "_via_img_metadata"
    if isinstance(data, dict) and "_via_img_metadata" in data:
        items = data["_via_img_metadata"].values()
    else:
        # sometimes directly a dict of entries
        items = data.values() if isinstance(data, dict) else data

    for item in items:
        filename = item.get("filename") or item.get("file_name")
        if not filename:
            continue

        # Match only if filename is in our sample folder
        # Some VIA files store path-like strings; keep basename
        filename_base = os.path.basename(filename)
        if filename_base not in sample_names:
            continue

        # Open image to get W,H
        img_path = IMG_OUT / filename_base
        if not img_path.exists():
            continue

        with Image.open(img_path) as im:
            w, h = im.size

        regions = item.get("regions", [])
        # In some VIA versions, regions is a dict; in others, list
        if isinstance(regions, dict):
            regions = list(regions.values())

        for r in regions:
            shape = r.get("shape_attributes", {})
            name = shape.get("name", "")

            # BarBeR polygons are usually name="polygon" with all_points_x/y
            xs = shape.get("all_points_x")
            ys = shape.get("all_points_y")
            if xs is None or ys is None:
                continue
            if len(xs) < 3 or len(ys) < 3:
                continue

            xmin, ymin, xmax, ymax = poly_to_bbox(xs, ys)

            # Clamp to image boundaries
            xmin = max(0, min(xmin, w-1))
            xmax = max(0, min(xmax, w-1))
            ymin = max(0, min(ymin, h-1))
            ymax = max(0, min(ymax, h-1))

            # Skip tiny / invalid boxes
            if xmax <= xmin or ymax <= ymin:
                continue

            xcn, ycn, bwn, bhn = bbox_to_yolo(xmin, ymin, xmax, ymax, w, h)
            labels_map[filename_base].append(f"{CLASS_ID} {xcn:.6f} {ycn:.6f} {bwn:.6f} {bhn:.6f}")
            kept_regions += 1

print("Total kept barcode regions:", kept_regions)


Reading VIA JSONs: 100%|██████████| 12/12 [00:14<00:00,  1.20s/it]

Total kept barcode regions: 877


In [6]:
#this is a check to see if there are empty jsons
empty = 0
for name, lines in labels_map.items():
    label_path = LBL_OUT / (Path(name).stem + ".txt")
    if len(lines) == 0:
        empty += 1
        #keep empty label files or skip them
        # For YOLO training many empties can hurt its training
        label_path.write_text("")
    else:
        label_path.write_text("\n".join(lines) + "\n")

print("Images with NO barcodes found in VIA annotations:", empty)
print("Label files:", len(list(LBL_OUT.glob("*.txt"))))#


Images with NO barcodes found in VIA annotations: 0
Label files: 789


In [7]:
#this create the data frame structure
data_yaml = OUT_DIR / "data.yaml"
data_yaml.write_text(f"""
path: {OUT_DIR}
train: images/train
val: images/val

names:
  0: barcode
""".strip() + "\n")

print(data_yaml.read_text())


path: /content/drive/MyDrive/BarBeR_yolo_sample
train: images/train
val: images/val

names:
  0: barcode



In [8]:
#SPLITTING IMAGES INTO TRAIN AND VAL AS THAT IS WHAT YOLO NEEDS TO WORK
from pathlib import Path
import random, shutil

BASE = Path("/content/drive/MyDrive/BarBeR_yolo_sample")
img_dir = BASE / "images"
lbl_dir = BASE / "labels"

# If you already have train/val folders, skip this check
print("Images folder contains:", list(img_dir.iterdir())[:5])
print("Labels folder contains:", list(lbl_dir.iterdir())[:5])

# Make train/val folders
(img_dir / "train").mkdir(parents=True, exist_ok=True)
(img_dir / "val").mkdir(parents=True, exist_ok=True)
(lbl_dir / "train").mkdir(parents=True, exist_ok=True)
(lbl_dir / "val").mkdir(parents=True, exist_ok=True)

# Collect images ONLY if they are directly inside images/ (not already in train/val)
imgs = sorted([p for p in img_dir.glob("*.jpg")])
print("Found flat images:", len(imgs))

# Adjust split ratio here
val_ratio = 0.2
random.seed(0)
random.shuffle(imgs)

n_val = int(len(imgs) * val_ratio)
val_imgs = imgs[:n_val]
train_imgs = imgs[n_val:]

def move_pair(img_path, split):
    # Move image
    dst_img = img_dir / split / img_path.name
    shutil.move(str(img_path), str(dst_img))

    # Move matching label (same filename but .txt)
    label_name = img_path.with_suffix(".txt").name
    src_lbl = lbl_dir / label_name
    dst_lbl = lbl_dir / split / label_name

    if src_lbl.exists():
        shutil.move(str(src_lbl), str(dst_lbl))
    else:
        # If there's no label file, create an empty one (YOLO expects a .txt even if no objects)
        dst_lbl.write_text("")

for p in train_imgs:
    move_pair(p, "train")

for p in val_imgs:
    move_pair(p, "val")

print("Done.")
print("train images:", len(list((img_dir/'train').glob('*.jpg'))))
print("val images:", len(list((img_dir/'val').glob('*.jpg'))))
print("train labels:", len(list((lbl_dir/'train').glob('*.txt'))))
print("val labels:", len(list((lbl_dir/'val').glob('*.txt'))))


Images folder contains: [PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/images/0020000111971.jpg'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/images/0011110669711.jpg'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/images/0022796916167.jpg'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/images/0045496472719_1.jpg'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/images/0071475200554.jpg')]
Labels folder contains: [PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/labels/train'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/labels/val'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/labels/train.cache'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/labels/val.cache'), PosixPath('/content/drive/MyDrive/BarBeR_yolo_sample/labels/4000576484736-01_N95.txt')]
Found flat images: 789
Done.
train images: 632
val images: 157
train labels: 632
val labels: 157


In [9]:
#training via YOLOv8
!pip -q install ultralytics

from ultralytics import YOLO

model = YOLO("yolov8n.pt") # small & fast; good for Colab

results = model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/drive/MyDrive/yolo_runs",
    name="barcode_train"
)
#saving best model

from google.colab import files
files.download("/content/drive/MyDrive/yolo_runs/barcode_train/weights/best.pt")


Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/BarBeR_yolo_sample/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=barcode_train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
val_folder = OUT_DIR / "images/val"
some = list(val_folder.glob("*.jpg"))[:10]
pred = model.predict(source=[str(p) for p in some], conf=0.25, save=True)
print("Saved predictions to:", pred[0].save_dir)



0: 640x640 1 barcode, 5.1ms
1: 640x640 1 barcode, 5.1ms
2: 640x640 1 barcode, 5.1ms
3: 640x640 1 barcode, 5.1ms
4: 640x640 1 barcode, 5.1ms
5: 640x640 1 barcode, 5.1ms
6: 640x640 1 barcode, 5.1ms
7: 640x640 1 barcode, 5.1ms
8: 640x640 1 barcode, 5.1ms
9: 640x640 1 barcode, 5.1ms
Speed: 2.6ms preprocess, 5.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict
Saved predictions to: /content/runs/detect/predict


In [12]:
!find /content -name "best.pt" -type f


^C


In [13]:
from ultralytics import YOLO

best_path = "/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.pt"
model = YOLO(best_path)
model.export(format="onnx", imgsz=320)   # 320 is better for Pi speed


Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 14 packages in 163ms
Prepared 6 packages in 4.85s
Installed 6 packages in 294ms
 + colorama==0.4.6
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnx==1.20.1
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.82

requirements: AutoUpdate success ✅ 5.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 7.1s, saved as '/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx' (11.6 MB)

Export complete (7.3s)
Results saved to /content/drive/MyDrive/yolo_runs/barcode_train2/weights
Predict:         yolo predict task=detect model=/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx imgsz=320 
Validate:        yolo val task=detect model=/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx imgsz=320 data=/content/drive/MyDrive/BarBeR_yolo_sample/data.yaml  
Visualize:       https://netron.app


'/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx'

In [14]:
!find /content/drive/MyDrive/yolo_runs/barcode_train2 -name "*.onnx" -type f


/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx


In [15]:
from google.colab import files
files.download("/content/drive/MyDrive/yolo_runs/barcode_train2/weights/best.onnx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>